In [2]:
import pandas as pd
import time
import os
import re
import google.generativeai as genai
from tqdm import tqdm
# from google.colab import drive

c:\Users\ACER\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# --- CONFIGURATION --- 
API_KEY = ""
MODEL_ID = "gemini-2.5-flash-lite"
START_ROW = 1
END_ROW = 1001

# File paths
INPUT_FILENAME = 'dataset_translated_1_1000.csv'
OUTPUT_FILENAME = 'dataset_translated_1_1000.csv'

genai.configure(api_key=API_KEY)

# Predefined list of tourist attraction aspects for ABSA
VALID_ASPECTS = [
    'place', 'spot', 'view', 'price', 'quietness', 'food', 'drink', 'maintenance', 
    'staff', 'peacefulness', 'road', 'air', 'water', 'comfort', 'photo', 'music', 
    'restroom', 'parking', 'taxi', 'spaciousness', 'light', 'accessibility', 
    'cleanliness', 'atmosphere', 'management', 'service', 'facility',
    'location', 'weather', 'crowd', 'safety', 'entrance', 'ticket', 'guide', 
    'information', 'wifi', 'internet', 'scenery', 'beauty', 'nature', 
    'architecture', 'history', 'culture', 'tradition', 'souvenir', 'shopping',
    'accommodation', 'hotel', 'restaurant', 'cafe', 'transportation', 
    'walking', 'hiking', 'trail', 'path', 'signage', 'direction', 'map',
    'security', 'noise', 'crowdedness', 'queue', 'waiting', 'opening_hours',
    'tour', 'activity', 'entertainment', 'attraction', 'monument', 'museum',
    'garden', 'beach', 'mountain', 'lake', 'river', 'bridge', 'temple',
    'religious', 'spiritual', 'calm', 'serenity', 'beauty', 'landscape'
]

def extract_aspects_with_real_words(text):
    """
    Extracts aspects, sentiments, and real descriptive words using Gemini API.
    Returns a tuple: (aspects_string, sentiments_string, real_aspects_string)
    All three are comma-separated strings with matching order and count.
    """
    max_retries = 5
    base_delay = 5

    for attempt in range(max_retries):
        try:
            model = genai.GenerativeModel(MODEL_ID)
            
            prompt = f"""Perform Aspect-Based Sentiment Analysis (ABSA) on the following tourist review text.

Extract aspects from this predefined list:
{', '.join(VALID_ASPECTS)}

For each aspect found:
1. Identify the aspect category from the predefined list
2. Determine the sentiment (positive, negative, or neutral)
3. Extract the EXACT word from the text that DIRECTLY represents or mentions this aspect (not adjectives describing it)

CRITICAL RULES:
- The number of aspects, sentiments, and real words MUST be exactly the same
- Each aspect must have exactly ONE corresponding sentiment and ONE corresponding real word
- Real words should be the NOUN or KEY WORD that represents the aspect itself, NOT adjectives or descriptions
- If aspect is "place", real word should be "place" (not "beautiful" or "awesome")
- If aspect is "view", real word should be "view" (not "beautiful" or "amazing")
- If aspect is "spot", real word should be "spot" (not descriptive adjectives)
- If the exact aspect word appears in text, use it; otherwise use the closest synonym
- Output format (MUST follow exactly):
  Aspects: aspect1, aspect2, aspect3
  Sentiments: sentiment1, sentiment2, sentiment3
  RealWords: word1, word2, word3

Examples:
Text: "The view is beautiful but the toilet queue is too long"
Aspects: view, restroom
Sentiments: positive, negative
RealWords: view, toilet

Text: "It's really nice to come here on Friday, so quiet 😁"
Aspects: atmosphere, quietness
Sentiments: positive, positive
RealWords: here, quiet

Text: "Awesome place, a beautiful spot for young people who like to sing"
Aspects: place, spot
Sentiments: positive, positive
RealWords: place, spot

Rules:
- Use only these sentiment values: positive, negative, or neutral
- Real words should be the aspect word itself or its direct synonym (1-2 words max)
- DO NOT use adjectives (beautiful, awesome, nice, great) as real words
- Ensure counts match: if you have 2 aspects, you must have 2 sentiments and 2 real words
- If no aspects found, output empty values:
  Aspects: 
  Sentiments: 
  RealWords: 
- Do not include explanations, only output the three lines above

Text: {text}

Output:"""
            
            response = model.generate_content(prompt)
            result = response.text.strip()
            
            # Parse the response
            if result.lower() == "none" or not result:
                return "", "", ""
            
            lines = [line.strip() for line in result.split('\n') if line.strip()]
            
            aspects_line = ""
            sentiments_line = ""
            real_words_line = ""
            
            for line in lines:
                line_lower = line.lower()
                if line_lower.startswith('aspects:'):
                    aspects_line = line.split(':', 1)[1].strip()
                elif line_lower.startswith('sentiments:'):
                    sentiments_line = line.split(':', 1)[1].strip()
                elif line_lower.startswith('realwords:') or line_lower.startswith('real words:') or line_lower.startswith('real_words:'):
                    real_words_line = line.split(':', 1)[1].strip()
            
            # Split and clean
            extracted_aspects = [a.strip().lower() for a in aspects_line.split(',') if a.strip()]
            extracted_sentiments = [s.strip().lower() for s in sentiments_line.split(',') if s.strip()]
            extracted_real_words = [w.strip() for w in real_words_line.split(',') if w.strip()]
            
            # Validate aspect matching with predefined list
            valid_aspects_lower = [va.lower() for va in VALID_ASPECTS]
            normalized_aspects = []
            
            for aspect in extracted_aspects:
                aspect_normalized = aspect.replace(' ', '_').replace('-', '_')
                if aspect_normalized in valid_aspects_lower:
                    idx = valid_aspects_lower.index(aspect_normalized)
                    normalized_aspects.append(VALID_ASPECTS[idx])
                elif aspect in valid_aspects_lower:
                    idx = valid_aspects_lower.index(aspect)
                    normalized_aspects.append(VALID_ASPECTS[idx])
            
            # Normalize sentiments
            normalized_sentiments = []
            for sentiment in extracted_sentiments:
                if sentiment in ['positive', 'pos', 'good', 'great', 'excellent']:
                    normalized_sentiments.append('positive')
                elif sentiment in ['negative', 'neg', 'bad', 'poor', 'terrible']:
                    normalized_sentiments.append('negative')
                else:
                    normalized_sentiments.append('neutral')
            
            # Ensure all three lists have the same length
            count = len(normalized_aspects)
            
            # Trim or pad sentiments
            if len(normalized_sentiments) > count:
                normalized_sentiments = normalized_sentiments[:count]
            elif len(normalized_sentiments) < count:
                normalized_sentiments.extend(['neutral'] * (count - len(normalized_sentiments)))
            
            # Trim or pad real words
            if len(extracted_real_words) > count:
                extracted_real_words = extracted_real_words[:count]
            elif len(extracted_real_words) < count:
                # If missing real words, use the aspect name as fallback
                for i in range(len(extracted_real_words), count):
                    if i < len(normalized_aspects):
                        extracted_real_words.append(normalized_aspects[i])
            
            if not normalized_aspects:
                return "", "", ""
            
            return (
                ', '.join(normalized_aspects),
                ', '.join(normalized_sentiments),
                ', '.join(extracted_real_words)
            )

        except Exception as e:
            error_str = str(e)
            
            if "503" in error_str or "overloaded" in error_str:
                wait_time = base_delay * (2 ** attempt)
                print(f"\n  ⚠️ Server overloaded (503). Retrying in {wait_time}s...")
                time.sleep(wait_time)
                continue
            
            elif "429" in error_str or "RESOURCE_EXHAUSTED" in error_str:
                wait_time = 4
                print(f"\n  ⚠️ Rate limit hit (429). Waiting {wait_time}s before retry...")
                time.sleep(wait_time)
                continue
            
            else:
                print(f"\n  ❌ Unexpected Error: {e}")
                return None, None, None

    return None, None, None

def main():
    # Ensure input file exists
    if not os.path.exists(INPUT_FILENAME):
        print(f"❌ Error: Could not find input file: {INPUT_FILENAME}")
        return

    # Resume Logic
    if os.path.exists(OUTPUT_FILENAME):
        print(f"🔄 Resuming from existing file: {OUTPUT_FILENAME}")
        df = pd.read_csv(OUTPUT_FILENAME)
    else:
        print(f"🆕 Starting fresh. Reading: {INPUT_FILENAME}")
        df = pd.read_csv(INPUT_FILENAME)
    
    # Ensure columns exist
    if 'aspects' not in df.columns:
        df['aspects'] = None
    if 'sentiment' not in df.columns:
        df['sentiment'] = None
    if 'real_aspect' not in df.columns:
        df['real_aspect'] = None

    # Convert to object type
    df['aspects'] = df['aspects'].astype(object)
    df['sentiment'] = df['sentiment'].astype(object)
    df['real_aspect'] = df['real_aspect'].astype(object)
    
    # Get missing rows
    missing_aspects = df[df['aspects'].isnull()].index
    missing_sentiments = df[df['sentiment'].isnull()].index
    missing_real_aspects = df[df['real_aspect'].isnull()].index
    missing_rows = missing_aspects.union(missing_sentiments).union(missing_real_aspects)
    
    # Apply START_ROW and END_ROW filter
    rows_to_process = [
        idx for idx in missing_rows 
        if idx >= START_ROW and (END_ROW is None or idx < END_ROW)
    ]
    
    print(f"Total rows in file: {len(df)}")
    print(f"Rows missing aspects, sentiment, or real_aspect: {len(missing_rows)}")
    print(f"Configuration: START_ROW={START_ROW}, END_ROW={END_ROW}")
    print(f"Actual rows to process: {len(rows_to_process)}")
    print(f"Extracting aspects, sentiments, and real aspects using Gemini...")
    print(f"Saving every 20 rows...")
    print("------------------------------------------------")

    count = 0
    
    for idx in tqdm(rows_to_process):
        text_column = 'english_translation' if 'english_translation' in df.columns else 'text'
        original_text = df.at[idx, text_column]
        
        if pd.isna(original_text) or str(original_text).strip() == "":
            df.at[idx, 'aspects'] = ""
            df.at[idx, 'sentiment'] = ""
            df.at[idx, 'real_aspect'] = ""
            continue

        aspects, sentiments, real_words = extract_aspects_with_real_words(original_text)

        if aspects is not None and sentiments is not None and real_words is not None:
            df.at[idx, 'aspects'] = aspects
            df.at[idx, 'sentiment'] = sentiments
            df.at[idx, 'real_aspect'] = real_words
        else:
            df.at[idx, 'aspects'] = ""
            df.at[idx, 'sentiment'] = ""
            df.at[idx, 'real_aspect'] = ""
        
        count += 1

        # Save every 20 rows
        if count % 20 == 0:
            df.to_csv(OUTPUT_FILENAME, index=False)
        
        # Rate Limit: 15 requests per minute = 4 seconds per request
        time.sleep(4)

    # Final Save
    df.to_csv(OUTPUT_FILENAME, index=False)
    print(f"✅ Process ended. Saved final file to: {OUTPUT_FILENAME}")

if __name__ == "__main__":
    main()

🔄 Resuming from existing file: dataset_translated_1_1000.csv
Total rows in file: 1000
Rows missing aspects, sentiment, or real_aspect: 35
Configuration: START_ROW=1, END_ROW=1001
Actual rows to process: 35
Extracting aspects, sentiments, and real aspects using Gemini...
Saving every 20 rows...
------------------------------------------------


  0%|          | 0/35 [00:00<?, ?it/s]

100%|██████████| 35/35 [02:45<00:00,  4.72s/it]

✅ Process ended. Saved final file to: dataset_translated_1_1000.csv
